# Week 5 Assignment: ARIA v3.0 - 全自動區域受災衝擊評估系統

## Captain's Log: 系統初始化

**任務目標**: 建立動態風險監測系統，整合 W3-W4 避難所資料與鳳凰颱風雨量數據

**技術架構**:
- 模式切換器 (LIVE/SIMULATION)
- CWA API 整合與 fallback 機制
- 動態風險疊合分析
- Folium 互動地圖輸出

**目標區域**: 花蓮縣 + 宜蘭縣
**測試情境**: 2025年鳳凰颱風 (fungwong_202511.json)

In [ ]:
# Import required libraries
import pandas as pd
import geopandas as gpd
import requests
import json
import numpy as np
import matplotlib.pyplot as plt
import folium
from folium import plugins
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')
import os
from shapely.geometry import Point
from pathlib import Path
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Configuration from .env
APP_MODE = os.getenv('APP_MODE', 'SIMULATION')
CWA_API_KEY = os.getenv('CWA_API_KEY')
SIMULATION_DATA = os.getenv('SIMULATION_DATA', 'data/fungwong_202511.json')
TARGET_COUNTIES = ["花蓮縣", "宜蘭縣"]
CRITICAL_RAINFALL_THRESHOLD = float(os.getenv('RAINFALL_CRITICAL_THRESHOLD', '80.0'))
URGENT_RAINFALL_THRESHOLD = float(os.getenv('RAINFALL_URGENT_THRESHOLD', '40.0'))
BUFFER_DISTANCE_KM = float(os.getenv('BUFFER_DISTANCE_KM', '5.0'))

print(f"Environment: {APP_MODE}")
print(f"Target Counties: {', '.join(TARGET_COUNTIES)}")
print(f"Critical Threshold: {CRITICAL_RAINFALL_THRESHOLD}mm/hr")
print(f"Urgent Threshold: {URGENT_RAINFALL_THRESHOLD}mm/hr")
print("Libraries loaded successfully!")

## Captain's Log: 資料獲取與標準化

**目標**: 實作模式切換器，統一處理 CWA API 與 CoLife 歷史資料格式

**挑戰**:
- CWA API 回傳兩組座標 (TWD67 + WGS84)
- CoLife 資料只有一組 WGS84 座標
- 數值型態差異 (string vs numeric)
- 需要過濾 -998 無資料值

In [ ]:
def normalize_cwa_json(raw_data):
    """
    統一處理 CWA API 與 CoLife 歷史資料格式
    
    Args:
        raw_data: 原始 JSON 資料
        
    Returns:
        dict: 標準化後的雨量資料
    """
    try:
        if not raw_data:
            return {'success': False, 'error': 'No data provided'}
        
        stations = []
        
        if 'records' in raw_data and 'Station' in raw_data['records']:
            station_records = raw_data['records']['Station']
            
            for record in station_records:
                try:
                    # 提取雨量資料
                    rainfall_element = record.get('RainfallElement', {})
                    rainfall_data = {
                        'Past1hr': float(rainfall_element.get('Past1hr', {}).get('Precipitation', 0)),
                        'Past24hr': float(rainfall_element.get('Past24hr', {}).get('Precipitation', 0))
                    }
                    
                    # 過濾 -998 無資料值
                    if rainfall_data['Past1hr'] == -998 or rainfall_data['Past24hr'] == -998:
                        continue
                    
                    # 提取座標 - 處理 CWA API 雙座標問題
                    coordinates = record.get('GeoInfo', {}).get('Coordinates', [])
                    lat = 0.0
                    lon = 0.0
                    
                    if coordinates and len(coordinates) > 0:
                        if len(coordinates) > 1:
                            # CWA API: 取第二組 WGS84 座標
                            coord = coordinates[1]
                        else:
                            # CoLife: 只有一組座標
                            coord = coordinates[0]
                        lat = float(coord.get('StationLatitude', 0))
                        lon = float(coord.get('StationLongitude', 0))
                    
                    # 建立站點資料
                    station_data = {
                        'StationId': record.get('StationId', ''),
                        'StationName': record.get('StationName', ''),
                        'CountyName': record.get('GeoInfo', {}).get('CountyName', ''),
                        'Latitude': lat,
                        'Longitude': lon,
                        'ObsTime': record.get('ObsTime', {}).get('DateTime', ''),
                        'Rainfall': rainfall_data
                    }
                    
                    stations.append(station_data)
                    
                except Exception as e:
                    print(f"Error processing station record: {e}")
                    continue
        
        return {
            'success': True,
            'total_count': len(stations),
            'stations': stations
        }
        
    except Exception as e:
        print(f"Error normalizing CWA data: {e}")
        return {'success': False, 'error': str(e)}

print("normalize_cwa_json() function defined")

In [ ]:
def get_rainfall_data():
    """
    根據 APP_MODE 獲取雨量資料
    """
    try:
        if APP_MODE == 'LIVE':
            print("🌐 Fetching live data from CWA API...")
            return fetch_cwa_live_data()
        else:
            print("📂 Loading simulation data from local file...")
            return load_simulation_data()
    except Exception as e:
        print(f"⚠️  Error in get_rainfall_data: {e}")
        print("🔄 Falling back to simulation data...")
        return load_simulation_data()

def fetch_cwa_live_data():
    """
    獲取 CWA 即時資料
    """
    try:
        api_url = "https://opendata.cwa.gov.tw/api/v1/rest/datastore/O-A0002-001"
        headers = {
            'Authorization': f'Bearer {CWA_API_KEY}',
            'Content-Type': 'application/json'
        }
        
        response = requests.get(api_url, headers=headers, timeout=10)
        response.raise_for_status()
        
        raw_data = response.json()
        normalized_data = normalize_cwa_json(raw_data)
        normalized_data['data_source'] = 'CWA_LIVE_API'
        
        print(f"✅ Successfully fetched {normalized_data.get('total_count', 0)} stations from CWA API")
        return normalized_data
        
    except requests.exceptions.RequestException as e:
        print(f"❌ CWA API request failed: {e}")
        raise
    except Exception as e:
        print(f"❌ Error processing CWA API data: {e}")
        raise

def load_simulation_data():
    """
    載入模擬資料
    """
    try:
        with open(SIMULATION_DATA, 'r', encoding='utf-8') as f:
            raw_data = json.load(f)
        
        normalized_data = normalize_cwa_json(raw_data)
        normalized_data['data_source'] = 'CWA_SIMULATION'
        
        print(f"✅ Successfully loaded {normalized_data.get('total_count', 0)} stations from simulation")
        return normalized_data
        
    except Exception as e:
        print(f"❌ Error loading simulation data: {e}")
        return {'success': False, 'error': str(e)}

print("Data acquisition functions defined")

In [ ]:
# 獲取雨量資料
print("=" * 50)
print("CWA Rainfall Data Analysis")
print("=" * 50)

rainfall_data = get_rainfall_data()

if rainfall_data['success']:
    print(f"\n✓ Successfully loaded data from {rainfall_data['data_source']}")
    print(f"✓ Total stations: {rainfall_data['total_count']}")
    
    # 篩選目標縣市
    target_stations = [
        station for station in rainfall_data['stations']
        if station['CountyName'] in TARGET_COUNTIES
    ]
    
    print(f"\nStations in target counties: {len(target_stations)}")
    
    if target_stations:
        # 排序顯示最高雨量站
        target_stations.sort(key=lambda x: x['Rainfall']['Past24hr'], reverse=True)
        
        print(f"\nTop 5 stations by 24hr rainfall:")
        for i, station in enumerate(target_stations[:5], 1):
            rainfall_24h = station['Rainfall']['Past24hr']
            county = station['CountyName']
            print(f"{i}. {station['StationName']} ({county}) - {rainfall_24h} mm")
else:
    print(f"\n✗ Failed to load data: {rainfall_data.get('error', 'Unknown error')}")

## Captain's Log: 空間資料處理

**目標**: 載入避難所與地形風險資料，建立空間分析基礎

**資料來源**:
- W3: 避難所位置與河川距離分級
- W4: 地形風險評估 (mean_elevation, max_slope, terrain_risk)

**座標系統**: EPSG:3826 (TWD97) 用於分析，EPSG:4326 用於視覺化

In [ ]:
def load_shelter_data():
    """
    載入避難所資料
    """
    try:
        shelters_df = pd.read_csv('data/避難收容處所.csv')
        print(f"Loaded {len(shelters_df)} shelters from CSV")
        
        # 篩選目標縣市
        target_shelters = shelters_df[
            shelters_df['縣市及鄉鎮市區'].str.contains('|'.join(TARGET_COUNTIES), na=False)
        ]
        print(f"Found {len(target_shelters)} shelters in target counties")
        
        # 建立 GeoDataFrame (注意順序: lon, lat)
        geometry = [Point(lon, lat) for lon, lat in zip(target_shelters['經度'], target_shelters['緯度'])]
        shelters_gdf = gpd.GeoDataFrame(target_shelters, geometry=geometry, crs='EPSG:4326')
        
        return shelters_gdf
        
    except Exception as e:
        print(f"Error loading shelter data: {e}")
        return None

def load_terrain_risk_data():
    """
    載入地形風險資料
    """
    try:
        with open('data/terrain_risk_audit.json', 'r', encoding='utf-8') as f:
            terrain_risk = json.load(f)
        
        print(f"Loaded terrain risk data for {len(terrain_risk)} shelters")
        
        # 轉換為 GeoDataFrame
        risk_df = pd.DataFrame(terrain_risk)
        geometry = [Point(coord['x'], coord['y']) for coord in risk_df['coordinates']]
        risk_gdf = gpd.GeoDataFrame(risk_df, geometry=geometry, crs='EPSG:3826')
        
        return risk_gdf
        
    except Exception as e:
        print(f"Error loading terrain risk data: {e}")
        return None

# 載入資料
shelters_gdf = load_shelter_data()
terrain_risk_gdf = load_terrain_risk_data()

In [ ]:
def create_rainfall_gdf(rainfall_data):
    """
    建立雨量站 GeoDataFrame
    """
    if not rainfall_data['success']:
        return None
    
    stations_df = pd.DataFrame(rainfall_data['stations'])
    
    # 篩選有效座標與目標縣市
    valid_stations = stations_df[
        (stations_df['Latitude'].notna()) & 
        (stations_df['Longitude'].notna()) &
        (stations_df['CountyName'].isin(TARGET_COUNTIES))
    ]
    
    print(f"Found {len(valid_stations)} valid rainfall stations in target counties")
    
    # 建立 GeoDataFrame (注意順序: lon, lat)
    geometry = [Point(lon, lat) for lon, lat in zip(valid_stations['Longitude'], valid_stations['Latitude'])]
    rainfall_gdf = gpd.GeoDataFrame(valid_stations, geometry=geometry, crs='EPSG:4326')
    
    return rainfall_gdf

# 建立雨量站 GeoDataFrame
rainfall_gdf = create_rainfall_gdf(rainfall_data)

if rainfall_gdf is not None:
    print(f"Rainfall stations CRS: {rainfall_gdf.crs}")
    print(f"Sample rainfall stations:")
    print(rainfall_gdf[['StationName', 'CountyName', 'Rainfall']].head())

## Captain's Log: 動態風險疊合分析

**目標**: 實作動態風險分級邏輯

**分析流程**:
1. 轉換 CRS 到 EPSG:3826 用於空間分析
2. 為高雨量站建立 5km 影響範圍 buffer
3. Spatial join 找出暴雨影響範圍內的避難所
4. 套用動態風險分級邏輯

**CRS 檢查**: 確保所有圖層座標系一致，避免 sjoin 結果為空

In [ ]:
def create_rainfall_impact_zones(rainfall_gdf):
    """
    建立高雨量站影響範圍 buffer
    """
    if rainfall_gdf is None:
        return None, None, None
    
    # 轉換到 EPSG:3826 (台灣座標系)
    rainfall_gdf_3826 = rainfall_gdf.to_crs('EPSG:3826')
    print(f"Converted rainfall stations to EPSG:3826: {rainfall_gdf_3826.crs}")
    
    # 篩選高雨量站
    high_rainfall_stations = rainfall_gdf_3826[
        rainfall_gdf_3826['Rainfall'].apply(lambda x: x.get('Past1hr', 0) > URGENT_RAINFALL_THRESHOLD)
    ]
    
    print(f"Found {len(high_rainfall_stations)} high rainfall stations (> {URGENT_RAINFALL_THRESHOLD}mm/hr)")
    
    # 建立 buffer
    buffer_distance = BUFFER_DISTANCE_KM * 1000  # km to meters
    high_rainfall_stations['buffer_geometry'] = high_rainfall_stations.geometry.buffer(buffer_distance)
    
    # 建立 buffer GeoDataFrame
    buffer_gdf = gpd.GeoDataFrame(
        high_rainfall_stations[['StationName', 'StationId', 'Rainfall']],
        geometry=high_rainfall_stations['buffer_geometry'],
        crs='EPSG:3826'
    )
    
    print(f"Created {len(buffer_gdf)} rainfall impact zones")
    return buffer_gdf, high_rainfall_stations, rainfall_gdf_3826

# 建立影響範圍
buffer_gdf, high_rainfall_stations, rainfall_gdf_3826 = create_rainfall_impact_zones(rainfall_gdf)

In [ ]:
def perform_risk_overlay_analysis(shelters_gdf, terrain_risk_gdf, buffer_gdf):
    """
    執行風險疊合分析
    """
    
    if shelters_gdf is None or terrain_risk_gdf is None:
        print("Missing required datasets for risk overlay analysis")
        return None
    
    # 轉換到 EPSG:3826
    shelters_gdf_3826 = shelters_gdf.to_crs('EPSG:3826')
    terrain_risk_gdf_3826 = terrain_risk_gdf.to_crs('EPSG:3826')
    
    # 處理無高雨量站情況
    if buffer_gdf is None or len(buffer_gdf) == 0:
        print("⚠️  No high rainfall stations found. All shelters classified as SAFE.")
        
        shelters_with_risk = shelters_gdf_3826.merge(
            terrain_risk_gdf_3826[['shelter_id', 'risk_level', 'name']], 
            left_on='避難收容處所名稱', 
            right_on='name',
            how='left'
        )
        
        shelters_with_risk['dynamic_risk'] = 'SAFE'
        shelters_with_risk['rainfall_1hr'] = 0.0
        return shelters_with_risk
    
    # CRS 一致性檢查
    assert str(shelters_gdf_3826.crs) == str(buffer_gdf.crs), "CRS MISMATCH between shelters and buffers!"
    assert str(terrain_risk_gdf_3826.crs) == str(buffer_gdf.crs), "CRS MISMATCH between terrain risk and buffers!"
    
    print("✓ CRS consistency check passed")
    
    # 執行 spatial join
    shelters_in_rainfall = gpd.sjoin(shelters_gdf_3826, buffer_gdf, how='inner', predicate='intersects')
    print(f"Found {len(shelters_in_rainfall)} shelters in rainfall impact zones")
    
    # 合併地形風險資料
    shelters_with_risk = shelters_in_rainfall.merge(
        terrain_risk_gdf_3826[['shelter_id', 'risk_level', 'name']], 
        left_on='避難收容處所名稱', 
        right_on='name',
        how='left'
    )
    
    print(f"Merged terrain risk data for shelters")
    return shelters_with_risk

# 執行風險疊合分析
shelters_with_risk = perform_risk_overlay_analysis(
    shelters_gdf, terrain_risk_gdf, buffer_gdf
)

In [ ]:
def classify_dynamic_risk(shelters_with_risk, high_rainfall_stations):
    """
    套用動態風險分級邏輯
    """
    
    if shelters_with_risk is None:
        return None
    
    # 檢查是否已全部分類為 SAFE
    if 'dynamic_risk' in shelters_with_risk.columns and shelters_with_risk['dynamic_risk'].eq('SAFE').all():
        print("All shelters classified as SAFE (no high rainfall detected)")
        return shelters_with_risk
    
    def get_station_rainfall(station_id):
        """獲取站點雨量"""
        if high_rainfall_stations is None or high_rainfall_stations.empty:
            return 0.0
        station_data = high_rainfall_stations[high_rainfall_stations['StationId'] == station_id]
        if not station_data.empty:
            return station_data.iloc[0]['Rainfall'].get('Past1hr', 0)
        return 0.0
    
    def classify_risk(row):
        """動態風險分級邏輯"""
        rainfall_1hr = get_station_rainfall(row['StationId'])
        terrain_risk = row.get('risk_level', '')
        
        # 地形風險判斷
        terrain_high = terrain_risk in ['極高風險', '高風險']
        
        # 動態風險分級
        if rainfall_1hr > CRITICAL_RAINFALL_THRESHOLD:
            return 'CRITICAL'
        elif rainfall_1hr > URGENT_RAINFALL_THRESHOLD and terrain_high:
            return 'URGENT'
        elif rainfall_1hr > URGENT_RAINFALL_THRESHOLD or terrain_high:
            return 'WARNING'
        else:
            return 'SAFE'
    
    # 套用風險分級
    shelters_with_risk['dynamic_risk'] = shelters_with_risk.apply(classify_risk, axis=1)
    shelters_with_risk['rainfall_1hr'] = shelters_with_risk['StationId'].apply(get_station_rainfall)
    
    # 風險分佈統計
    risk_counts = shelters_with_risk['dynamic_risk'].value_counts()
    print("Dynamic Risk Classification Results:")
    print("=" * 40)
    for risk_level, count in risk_counts.items():
        print(f"{risk_level:10s}: {count:3d} shelters")
    
    return shelters_with_risk

# 套用動態風險分級
classified_shelters = classify_dynamic_risk(shelters_with_risk, high_rainfall_stations)

if classified_shelters is not None:
    print(f"\nSample classified shelters:")
    print(classified_shelters[['避難收容處所名稱', 'rainfall_1hr', 'risk_level', 'dynamic_risk']].head())

## Captain's Log: Folium 互動地圖

**目標**: 建立專業的互動式風險監測地圖

**圖層設計**:
- 雨量站 CircleMarker (半徑 ∝ 雨量，顏色分級)
- 避難所 Marker (依動態風險等級著色)
- HeatMap 雨量分佈強度
- LayerControl 圖層切換

**座標注意**: Folium 使用 [lat, lon] 順序

In [ ]:
def create_risk_map(classified_shelters, rainfall_gdf_3826, buffer_gdf):
    """
    建立互動風險地圖
    """
    if classified_shelters is None or rainfall_gdf_3826 is None:
        print("Missing data for map creation")
        return None
    
    # 計算地圖中心 (花蓮縣中心點)
    center_lat = 23.8
    center_lon = 121.3
    
    # 建立地圖
    m = folium.Map(
        location=[center_lat, center_lon],
        zoom_start=10,
        tiles='OpenStreetMap'
    )
    
    # 風險等級顏色
    risk_colors = {
        'CRITICAL': 'red',
        'URGENT': 'orange', 
        'WARNING': 'yellow',
        'SAFE': 'green'
    }
    
    # 新增避難所圖層
    for idx, shelter in classified_shelters.iterrows():
        risk_level = shelter['dynamic_risk']
        color = risk_colors.get(risk_level, 'blue')
        
        # 注意 Folium 使用 [lat, lon] 順序
        lat = shelter.geometry.y
        lon = shelter.geometry.x
        
        popup_text = f"""
        <b>{shelter['避難收容處所名稱']}</b><br>
        地形風險: {shelter.get('risk_level', 'N/A')}<br>
        動態風險: {risk_level}<br>
        時雨量: {shelter.get('rainfall_1hr', 0):.1f}mm<br>
        影響站點: {shelter.get('StationName', 'N/A')}<br>
        收容容量: {shelter.get('預計收容人數', 0)} 人
        """
        
        folium.Marker(
            location=[lat, lon],
            popup=folium.Popup(popup_text, max_width=300),
            icon=folium.Icon(color=color, icon='info-sign')
        ).add_to(m)
    
    # 新增雨量站圖層
    for idx, station in rainfall_gdf_3826.iterrows():
        rainfall_1hr = station['Rainfall']['Past1hr']
        rainfall_24hr = station['Rainfall']['Past24hr']
        
        # 雨量顏色分級
        if rainfall_1hr > CRITICAL_RAINFALL_THRESHOLD:
            color = 'red'
        elif rainfall_1hr > URGENT_RAINFALL_THRESHOLD:
            color = 'orange'
        elif rainfall_1hr > 0:
            color = 'yellow'
        else:
            color = 'green'
        
        # 半徑與雨量成正比
        radius = min(5 + rainfall_1hr * 0.5, 20)
        
        # 注意 Folium 使用 [lat, lon] 順序
        lat = station.geometry.y
        lon = station.geometry.x
        
        folium.CircleMarker(
            location=[lat, lon],
            radius=radius,
            popup=f"{station['StationName']}<br>時雨量: {rainfall_1hr:.1f}mm<br>24hr: {rainfall_24hr:.1f}mm",
            color=color,
            fill=True,
            fillColor=color,
            fillOpacity=0.7
        ).add_to(m)
    
    # 新增 HeatMap (注意座標順序 [lat, lon])
    heat_data = []
    for idx, station in rainfall_gdf_3826.iterrows():
        lat = station.geometry.y
        lon = station.geometry.x
        weight = station['Rainfall']['Past1hr']
        if weight > 0:  # 只加入有雨量的站點
            heat_data.append([lat, lon, weight])
    
    if heat_data:
        plugins.HeatMap(
            heat_data,
            name='Rainfall HeatMap',
            radius=15,
            blur=10,
            gradient={0.2: 'blue', 0.4: 'cyan', 0.6: 'yellow', 0.8: 'orange', 1.0: 'red'}
        ).add_to(m)
    
    # 新增圖層控制
    folium.LayerControl().add_to(m)
    
    # 新增圖例
    legend_html = '''
    <div style="position: fixed; 
                bottom: 50px; left: 50px; width: 150px; height: 120px; 
                background-color: white; border:2px solid grey; z-index:9999; 
                font-size:14px; padding: 10px">
    <h4>風險等級</h4>
    <i class="fa fa-circle" style="color:red"></i> CRITICAL<br>
    <i class="fa fa-circle" style="color:orange"></i> URGENT<br>
    <i class="fa fa-circle" style="color:yellow"></i> WARNING<br>
    <i class="fa fa-circle" style="color:green"></i> SAFE
    </div>
    '''
    m.get_root().html.add_child(folium.Element(legend_html))
    
    return m

print("Map creation function defined")

In [ ]:
# 建立互動地圖
print("Creating interactive risk map...")

risk_map = create_risk_map(classified_shelters, rainfall_gdf_3826, buffer_gdf)

if risk_map is not None:
    # 確保 outputs 目錄存在
    os.makedirs('outputs', exist_ok=True)
    
    # 儲存地圖
    map_filename = 'outputs/ARIA_v3_Fungwong.html'
    risk_map.save(map_filename)
    
    print(f"✅ Interactive risk map saved to: {map_filename}")
    print(f"🗺️  Map features:")
    print(f"   • {len(classified_shelters)} shelters with dynamic risk classification")
    print(f"   • {len(rainfall_gdf_3826)} rainfall stations")
    print(f"   • HeatMap showing rainfall intensity")
    print(f"   • Layer control for interactive exploration")
else:
    print("❌ Failed to create risk map")

## Captain's Log: 分析結果摘要

**任務完成**: ARIA v3.0 動態風險監測系統已成功建立

**系統特色**:
- ✅ 模式切換器 (LIVE/SIMULATION)
- ✅ 動態風險分級 (CRITICAL/URGENT/WARNING/SAFE)
- ✅ 專業 Folium 互動地圖
- ✅ 完整的 CRS 處理與空間分析
- ✅ 防呆檢查與錯誤處理機制

In [ ]:
# 最終分析摘要
print("=" * 80)
print("ARIA v3.0 - 動態風險監測系統")
print("=" * 80)
print(f"Analysis completed: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Environment: {APP_MODE}")
print()

# 計算風險統計
if classified_shelters is not None:
    risk_summary = classified_shelters['dynamic_risk'].value_counts()
    total_shelters = len(classified_shelters)
    
    print(f"📊 ANALYSIS SUMMARY:")
    print(f"   • Total shelters analyzed: {total_shelters}")
    print(f"   • Critical risk shelters: {risk_summary.get('CRITICAL', 0)}")
    print(f"   • Urgent risk shelters: {risk_summary.get('URGENT', 0)}")
    print(f"   • Warning risk shelters: {risk_summary.get('WARNING', 0)}")
    print(f"   • Safe shelters: {risk_summary.get('SAFE', 0)}")
    print()

print(f"🗺️  INTERACTIVE MAP:")
print(f"   • Professional risk visualization created")
print(f"   • Dynamic risk classification applied")
print(f"   • Rainfall impact zones displayed")
print(f"   • Saved as: outputs/ARIA_v3_Fungwong.html")
print()

print(f"🔧 DYNAMIC RISK CLASSIFICATION:")
print(f"   • CRITICAL: Rainfall > {CRITICAL_RAINFALL_THRESHOLD}mm/hr")
print(f"   • URGENT: Rainfall > {URGENT_RAINFALL_THRESHOLD}mm/hr + High terrain risk")
print(f"   • WARNING: Rainfall > {URGENT_RAINFALL_THRESHOLD}mm/hr OR High terrain risk")
print(f"   • SAFE: Below all thresholds")
print()

# 緊急警報檢查
if classified_shelters is not None:
    critical_count = risk_summary.get('CRITICAL', 0)
    urgent_count = risk_summary.get('URGENT', 0)
    
    if critical_count > 0:
        print("🚨 EMERGENCY ALERT:")
        print(f"   • {critical_count} shelters require IMMEDIATE attention")
        print(f"   • Rainfall exceeds critical threshold ({CRITICAL_RAINFALL_THRESHOLD}mm/hr)")
        print(f"   • Emergency response teams should be activated")
    elif urgent_count > 0:
        print("⚠️  HIGH PRIORITY ALERT:")
        print(f"   • {urgent_count} shelters require urgent monitoring")
        print(f"   • Combined rainfall and terrain risk detected")
    else:
        print("✅ STATUS: All shelters operating within safe parameters")
else:
    print("⚠️  No classified shelter data available")

print()
print("=" * 80)
print("ARIA v3.0 SYSTEM SUCCESSFULLY DEPLOYED")
print("=" * 80)